# Shopee ETL trên Jupyter / Google Colab

Notebook này chạy toàn bộ pipeline **Extract → Transform → Load** mà không cần Docker:

| Thành phần | Bản Docker | Bản notebook |
|---|---|---|
| Data lake | MinIO | thư mục `data_lake/` |
| Kho dữ liệu | PostgreSQL | file SQLite `shopee_dw.db` |
| Điều phối | Airflow | chạy từng cell |

Code ETL dùng chung với bản Docker (thư mục `src/shopee_etl`), nên những gì bạn học ở đây áp dụng y nguyên khi chuyển sang Airflow.

**Cách dùng:**
- **Jupyter trên máy**: đặt notebook này trong thư mục `shopee-etl` rồi chạy lần lượt từng cell.
- **Google Colab**: mở notebook, chạy cell đầu tiên, rồi tải file `shopee-etl.zip` lên khi được hỏi.

## 1. Chuẩn bị môi trường

In [ ]:
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("shopee-etl").exists():
        from google.colab import files
        print("Chọn file shopee-etl.zip để tải lên...")
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall(".")
    PROJECT = Path("shopee-etl").resolve()
else:
    PROJECT = Path.cwd()
    if not (PROJECT / "src" / "shopee_etl").exists() and (PROJECT / "shopee-etl").exists():
        PROJECT = PROJECT / "shopee-etl"

assert (PROJECT / "src" / "shopee_etl").exists(), f"Không thấy thư mục src/shopee_etl trong {PROJECT}"
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
print("Thư mục project:", PROJECT)

In [ ]:
# Colab đã có sẵn pandas, requests, pyarrow. Lệnh này chỉ cài phần còn thiếu.
%pip install -q pandas pyarrow requests matplotlib

## 2. Cấu hình

Để `SHOPEE_MOCK = "true"` để chạy bằng dữ liệu giả lập. Khi đã có App ID và Secret từ Shopee Affiliate, điền vào và đổi thành `"false"`.

Trên Colab nên lưu key trong mục **Secrets** (biểu tượng chìa khóa bên trái) thay vì gõ thẳng vào notebook.

In [ ]:
os.environ.update({
    "SHOPEE_MOCK": "true",
    "SHOPEE_APP_ID": "",          # điền App ID khi có
    "SHOPEE_APP_SECRET": "",      # điền App Secret khi có
    "SHOPEE_KEYWORDS": "tai nghe bluetooth,ban phim co,chuot gaming",
    "SHOPEE_PAGE_LIMIT": "50",
    "SHOPEE_MAX_PAGES": "2",
    "STORAGE_BACKEND": "local",   # lưu vào thư mục data_lake/ thay cho MinIO
    "LOCAL_LAKE_DIR": "data_lake",
})

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["SHOPEE_APP_ID"] = userdata.get("SHOPEE_APP_ID") or ""
        os.environ["SHOPEE_APP_SECRET"] = userdata.get("SHOPEE_APP_SECRET") or ""
        if os.environ["SHOPEE_APP_ID"]:
            os.environ["SHOPEE_MOCK"] = "false"
    except Exception:
        pass  # chưa tạo Secret thì tiếp tục chạy mock

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from shopee_etl.config import get_settings
from shopee_etl.storage import LakeStorage

settings = get_settings()
storage = LakeStorage(settings)
print("Chế độ mock:", settings.mock, "| Từ khóa:", settings.keywords)

## 3. Extract: lấy dữ liệu và lưu bản gốc

Dữ liệu được lưu nguyên dạng JSON, chia thư mục theo ngày và từ khóa. Giữ bản gốc giúp bạn chạy lại bước transform bất cứ lúc nào mà không phải gọi API lại.

In [ ]:
import json
from datetime import date, timedelta
from shopee_etl.extract import run_extract, raw_prefix

today = date.today()
n = run_extract(settings, storage, today)

keys = storage.list_keys(raw_prefix(today))
print(f"Lấy được {n} sản phẩm, lưu thành {len(keys)} file:")
for k in keys:
    print("  ", k)

In [ ]:
# Xem thử một bản ghi gốc
sample = json.loads(storage.get_bytes(keys[0]))
print(json.dumps(sample["nodes"][0], ensure_ascii=False, indent=2))

## 4. Transform: làm sạch dữ liệu

Bước này làm phẳng JSON thành bảng, ép kiểu, bỏ trùng và kiểm tra chất lượng. Kết quả được ghi ra file Parquet.

In [ ]:
from shopee_etl.transform import run_transform, processed_key

run_transform(storage, today)
df = storage.get_parquet(processed_key(today))
print(df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df[["price", "discount_rate", "sales", "rating_star", "commission_rate"]].describe().round(2)

## 5. Load: nạp vào kho dữ liệu SQLite

Notebook dùng SQLite thay cho PostgreSQL, với cùng star schema trong `sql/schema.sql` (chỉ bỏ vài cú pháp riêng của PostgreSQL). Hàm `load_dataframe` cũng chính là hàm bản Docker dùng.

In [ ]:
import sqlite3
from shopee_etl.load import load_dataframe

DB_PATH = "shopee_dw.db"
conn = sqlite3.connect(DB_PATH)

schema = Path("sql/schema.sql").read_text().split("CREATE OR REPLACE VIEW")[0]
schema = schema.replace("TIMESTAMPTZ NOT NULL DEFAULT now()", "TEXT DEFAULT CURRENT_TIMESTAMP")
conn.executescript(schema)

load_dataframe(conn, df, today, placeholder="?")

for table in ["dim_date", "dim_shop", "dim_product", "fact_product_daily"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:20s} {count:>6} dòng")

## 6. Mô phỏng pipeline chạy hằng ngày

Airflow sẽ chạy pipeline mỗi sáng. Ở đây ta giả lập 7 ngày gần nhất để có dữ liệu lịch sử phân tích biến động giá.

Ô này chỉ chạy ở chế độ mock: API thật chỉ trả dữ liệu hiện tại, không lấy lại được giá của ngày đã qua.

In [ ]:
if settings.mock:
    for i in range(7, 0, -1):
        d = today - timedelta(days=i)
        run_extract(settings, storage, d)
        run_transform(storage, d)
        load_dataframe(conn, storage.get_parquet(processed_key(d)), d, placeholder="?")
    print("Số ngày trong kho:",
          conn.execute("SELECT COUNT(DISTINCT snapshot_date) FROM fact_product_daily").fetchone()[0])
else:
    print("Đang dùng API thật: hãy chạy lại notebook mỗi ngày để tích lũy lịch sử.")

## 7. Phân tích

In [ ]:
import pandas as pd

def sql(query):
    return pd.read_sql_query(query, conn)

# Biến động giá và lượt bán so với ngày trước đó
PRICE_CHANGE = '''
SELECT f.snapshot_date, f.item_id, p.product_name, s.shop_name, f.price, f.sales,
       LAG(f.price) OVER (PARTITION BY f.item_id ORDER BY f.snapshot_date) AS prev_price,
       f.sales - LAG(f.sales) OVER (PARTITION BY f.item_id ORDER BY f.snapshot_date) AS sales_gained
FROM fact_product_daily f
JOIN dim_product p USING (item_id)
JOIN dim_shop    s ON s.shop_id = f.shop_id
'''

latest = sql("SELECT MAX(snapshot_date) AS d FROM fact_product_daily")["d"][0]

print("Sản phẩm giảm giá mạnh nhất ngày", latest)
sql(f'''
SELECT product_name, shop_name, prev_price, price, price - prev_price AS price_diff
FROM ({PRICE_CHANGE})
WHERE snapshot_date = '{latest}' AND price < prev_price
ORDER BY price_diff LIMIT 10
''')

In [ ]:
print("Giá trung bình theo loại shop")
sql(f'''
SELECT s.shop_type, ROUND(AVG(f.price)) AS gia_tb, ROUND(AVG(f.rating_star), 2) AS rating_tb, COUNT(*) AS so_sp
FROM fact_product_daily f JOIN dim_shop s USING (shop_id)
WHERE f.snapshot_date = '{latest}'
GROUP BY s.shop_type ORDER BY gia_tb DESC
''')

In [ ]:
import matplotlib.pyplot as plt

history = sql(PRICE_CHANGE)
history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

# 5 sản phẩm bán chạy nhất ở ngày gần nhất
top = (history[history["snapshot_date"] == history["snapshot_date"].max()]
       .nlargest(5, "sales")["item_id"])

fig, ax = plt.subplots(figsize=(10, 5))
for item_id, g in history[history["item_id"].isin(top)].groupby("item_id"):
    ax.plot(g["snapshot_date"], g["price"] / 1000, marker="o", label=g["product_name"].iloc[0][:35])
ax.set_title("Biến động giá của 5 sản phẩm bán chạy nhất")
ax.set_ylabel("Giá (nghìn đồng)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
sold = (history.dropna(subset=["sales_gained"])
        .groupby("product_name")["sales_gained"].sum()
        .nlargest(10).sort_values())

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([name[:40] for name in sold.index], sold.values)
ax.set_title("Top 10 sản phẩm tăng lượt bán nhiều nhất trong kỳ")
ax.set_xlabel("Số lượt bán tăng thêm")
plt.tight_layout()
plt.show()

## 8. Bước tiếp theo

- **Dùng dữ liệu thật**: điền App ID và Secret ở mục 2, đổi `SHOPEE_MOCK` thành `"false"`, rồi chạy lại mục 3 đến 5 mỗi ngày.
- **Tải kết quả về**: thư mục `data_lake/` chứa dữ liệu gốc và Parquet, file `shopee_dw.db` mở được bằng DBeaver hoặc DB Browser for SQLite.
- **Chuyển sang Airflow**: khi đã quen, chạy bản Docker trong `README.md`. Code ETL giữ nguyên, chỉ đổi lưu trữ sang MinIO và kho sang PostgreSQL.

Lưu ý khi dùng Colab: dữ liệu trong phiên sẽ mất khi Colab ngắt kết nối. Muốn giữ lại, hãy mount Google Drive và đặt `LOCAL_LAKE_DIR` cùng `DB_PATH` vào thư mục trên Drive.